# Experiment 4: Optuna Hyperparameter Search — MLP on FashionMNIST

## HP Discovery + Bias Sweep — End-to-End

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Setup | Import libraries, configure paths, detect device | `src/train_utils.py`, `src/eval_utils.py` |
| 2 | Load Dataset | FashionMNIST with train/val/test split | `torchvision.datasets` |
| 3 | Define Model Builder | Dynamic MLP with searchable HPs + BatchNorm | — |
| 4 | Define Objective | Optuna objective with PR-AUC pruning | `src/train_utils.py`, `src/eval_utils.py` |
| 5 | Run Hyperparameter Search | 30-trial TPE search, 30 epochs/trial, Median pruner | `optuna` |
| 6 | Report Best Trial | Print best config, save all_trials.csv | — |
| 7 | Retrain Best Config | 30 epochs, best-checkpoint, dynamic batch_size | `src/train_utils.py` |
| 8 | Evaluate Best Model | Test accuracy, confusion matrix, per-class metrics | `src/eval_utils.py` |
| 9 | Logit Bias Sweep | Sweep logit bias for Shirt class trade-off | — |
| 10 | Cross-Experiment Comparison | Accuracy / Shirt TPR / Precision vs Phase 1–3 | — |

---




In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import optuna
from optuna.trial import TrialState
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score

def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

from src.train_utils import train_one_epoch
from src.eval_utils import (
    get_all_probas_and_labels, compute_pr_auc_scores, evaluate_detailed
)

DEVICE = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available()
          else 'cpu')
print(f'Device: {DEVICE}')

OUT_DIR = os.path.join(PROJ_ROOT, 'outputs', 'error_analysis', 'MLP', 'phase4_optuna')
DATA_DIR = os.path.join(PROJ_ROOT, 'data')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Search config ────────────────────────────────────────────────────────────
N_TRIALS = 30                # reduced from 50 to account for 2x epochs/trial
N_VALID = 5000               # held-out validation samples
EPOCHS_PER_TRIAL = 30        # matches final retrain length (fix: 15->30 epoch disconnect)
BATCH_SIZE = 256
FIXED_TEST_BATCH = 512

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'OUT_DIR: {OUT_DIR}')
print(f'Trials: {N_TRIALS}  Epochs/trial: {EPOCHS_PER_TRIAL}')



Device: cuda
OUT_DIR: c:\document\Study documents\Deeplearning_Course\outputs\error_analysis\MLP\phase4_optuna
Trials: 50  Epochs/trial: 15


## Dataset — with held-out validation split

All trials share the same fixed 5 000-sample validation split.
TPE may overfit to this split's quirks — the test-set evaluation is the real check.


In [2]:
_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
_train_full = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=False, transform=_tf)
_test_full = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=False, transform=_tf)

_indices = np.arange(len(_train_full))
np.random.seed(42)
np.random.shuffle(_indices)
_train_subset = Subset(_train_full, _indices[N_VALID:])
_val_subset = Subset(_train_full, _indices[:N_VALID])

TRAIN_LOADER = DataLoader(_train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
VAL_LOADER = DataLoader(_val_subset, batch_size=FIXED_TEST_BATCH, shuffle=False,
                        num_workers=0, pin_memory=True)
TEST_LOADER = DataLoader(_test_full, batch_size=FIXED_TEST_BATCH, shuffle=False,
                         num_workers=0, pin_memory=True)

print(f'Train: {len(TRAIN_LOADER)} batches  Val: {len(VAL_LOADER)} batches  Test: {len(TEST_LOADER)} batches')


Train: 215 batches  Val: 10 batches  Test: 20 batches


## Dynamic MLP builder

Search space: 1–4 hidden layers, 64–1024 units (log-uniform), 4 activations, dropout 0–0.5, BatchNorm placement `{none, before_act, after_act}`.



In [3]:
ACTIVATIONS = {
    'ReLU': lambda: nn.ReLU(),
    'LeakyReLU': lambda: nn.LeakyReLU(0.1),
    'ELU': lambda: nn.ELU(alpha=1.0),
    'GELU': lambda: nn.GELU(),
}

def build_mlp(trial):
    n_layers = trial.suggest_int('n_layers', 1, 4)
    units = []
    for i in range(n_layers):
        low, high = (128, 1024) if n_layers <= 3 else (64, 512)
        units.append(trial.suggest_int(f'units_{i}', low, high, log=True))
    act_name = trial.suggest_categorical('activation', list(ACTIVATIONS.keys()))
    act_cls = ACTIVATIONS[act_name]
    dropout = trial.suggest_float('dropout', 0.0, 0.5, step=0.05)
    batch_norm = trial.suggest_categorical('batch_norm', ['none', 'before_act', 'after_act'])
    layers = [nn.Flatten()]
    in_dim = 784
    for i, out_dim in enumerate(units):
        layers.append(nn.Linear(in_dim, out_dim))
        if batch_norm == 'before_act':
            layers.append(nn.BatchNorm1d(out_dim))
        layers.append(act_cls())
        if batch_norm == 'after_act':
            layers.append(nn.BatchNorm1d(out_dim))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_dim = out_dim
    layers.append(nn.Linear(in_dim, 10))
    return nn.Sequential(*layers)



## Optuna search — objective function

**Pruning & ranking metric**: macro PR-AUC (consistent across both).
**Validation accuracy** logged as user attribute for reference.
**New HPs**: batch_size (64/128/256/512), BatchNorm placement.
**Bug fix**: EPOCHS_PER_TRIAL = 30 (matches final retrain length).



In [4]:
def objective(trial):
    torch.manual_seed(42)
    if DEVICE.startswith('cuda'):
        torch.cuda.manual_seed_all(42)

    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])
    train_loader = DataLoader(_train_subset, batch_size=batch_size,
                              shuffle=True, num_workers=0, pin_memory=True)

    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    optim_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])
    scheduler_on = trial.suggest_categorical('scheduler', ['none', 'cosine', 'step'])

    model = build_mlp(trial).to(DEVICE)
    if optim_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif optim_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        momentum = trial.suggest_float('momentum', 0.8, 0.99)
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=momentum)

    criterion = nn.CrossEntropyLoss()
    if scheduler_on == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_TRIAL)
    elif scheduler_on == 'step':
        step_size = trial.suggest_int('step_size', 5, 10)
        gamma = trial.suggest_float('gamma', 0.1, 0.5)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    else:
        scheduler = None

    for epoch in range(EPOCHS_PER_TRIAL):
        loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        if scheduler:
            scheduler.step()
        probas, labels = get_all_probas_and_labels(model, VAL_LOADER, DEVICE, 10)
        pr_scores = compute_pr_auc_scores(probas, labels, model_name='_trial')
        val_pr = pr_scores['macro']
        trial.report(val_pr, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in VAL_LOADER:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    trial.set_user_attr('val_acc', 100 * correct / total)

    return val_pr



## Run the study


In [5]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, n_warmup_steps=3, interval_steps=1
    ),
    study_name='mlp_fashionmnist',
    storage=f'sqlite:///{OUT_DIR}/optuna_study.db',
)
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)



[I 2026-07-27 09:27:14,177] A new study created in memory with name: mlp_fashionmnist


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-27 09:30:06,434] Trial 0 finished with value: 0.9522478927220386 and parameters: {'lr': 0.0005611516415334506, 'weight_decay': 0.0007114476009343421, 'optimizer': 'Adam', 'scheduler': 'step', 'n_layers': 3, 'units_0': 558, 'units_1': 133, 'units_2': 962, 'activation': 'ReLU', 'dropout': 0.15000000000000002, 'step_size': 8, 'gamma': 0.2727780074568463}. Best is trial 0 with value: 0.9522478927220386.
[I 2026-07-27 09:34:07,116] Trial 1 finished with value: 0.6264387500023365 and parameters: {'lr': 0.0003823475224675188, 'weight_decay': 6.847920095574779e-05, 'optimizer': 'SGD', 'scheduler': 'cosine', 'n_layers': 3, 'units_0': 438, 'units_1': 140, 'units_2': 452, 'activation': 'GELU', 'dropout': 0.4, 'momentum': 0.8578766161429404}. Best is trial 0 with value: 0.9522478927220386.
[I 2026-07-27 09:36:52,518] Trial 2 finished with value: 0.6267542413936912 and parameters: {'lr': 0.0001567993391672301, 'weight_decay': 0.00011290133559092664, 'optimizer': 'SGD', 'scheduler': 'cosi

In [6]:
pruned = sum(1 for t in study.trials if t.state == TrialState.PRUNED)
complete = sum(1 for t in study.trials if t.state == TrialState.COMPLETE)
print(f'Study completed: {complete} complete, {pruned} pruned')

df = study.trials_dataframe()
df.to_csv(os.path.join(OUT_DIR, 'all_trials.csv'), index=False)

best = study.best_trial
print(f'\n{"="*70}')
print(f'Best trial: #{best.number}')
print(f'Val macro PR-AUC: {best.value:.6f}')
print(f'Val accuracy: {best.user_attrs.get("val_acc", "N/A"):.2f}%')
print(f'Params:')
for k, v in best.params.items():
    print(f'  {k}: {v}')


Study completed: 29 complete, 21 pruned

Best trial: #31
Val macro PR-AUC: 0.959261
Val accuracy: 90.08%
Params:
  lr: 0.002160220251328714
  weight_decay: 2.0239797085682152e-06
  optimizer: AdamW
  scheduler: cosine
  n_layers: 2
  units_0: 203
  units_1: 457
  activation: GELU
  dropout: 0.05


## Best config — retrain 30 epochs with best-checkpoint early stopping

Reconstructs from the best HP configuration, trains 30 epochs tracking validation accuracy,
and restores the best checkpoint before final test evaluation.


In [7]:
# ── Rebuild model from best params ──────────────────────────────────────────
best_bs = best.params.get('batch_size', 256)
train_loader = DataLoader(_train_subset, batch_size=best_bs,
                          shuffle=True, num_workers=0, pin_memory=True)

class FixedTrial:
    def __init__(self, params):
        self._params = params
    def suggest_int(self, name, low, high, log=False):
        return self._params[name]
    def suggest_float(self, name, low, high, log=False, step=None):
        return self._params[name]
    def suggest_categorical(self, name, choices):
        return self._params[name]

trial_stub = FixedTrial(best.params)
model = build_mlp(trial_stub).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

optim_name = best.params['optimizer']
lr = best.params['lr']
wd = best.params['weight_decay']
if optim_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
elif optim_name == 'AdamW':
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
else:
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=wd,
                          momentum=best.params['momentum'])

scheduler = None
if best.params['scheduler'] == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
elif best.params['scheduler'] == 'step':
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, step_size=best.params['step_size'],
        gamma=best.params['gamma'])

criterion = nn.CrossEntropyLoss()

# ── Train 30 epochs with best-val checkpoint ────────────────────────────────
print(f'\nTraining best config for 30 epochs...')
train_losses = []
best_val_acc = -1.0
best_epoch = -1
for epoch in range(30):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    train_losses.append(loss)
    if scheduler:
        scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in VAL_LOADER:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
    val_acc_epoch = 100 * correct / total

    if val_acc_epoch > best_val_acc:
        best_val_acc = val_acc_epoch
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_checkpoint.pth'))

    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == 29:
        print(f'  Epoch [{epoch+1}/30] Loss: {loss:.4f}  ValAcc: {val_acc_epoch:.2f}%')

print(f'  Best checkpoint at epoch {best_epoch+1}: val_acc={best_val_acc:.2f}%')
model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'best_checkpoint.pth')))
model.to(DEVICE)



Params: 257,163

Training best config for 30 epochs...
  Epoch [1/30] Loss: 0.5252  ValAcc: 85.68%
  Epoch [5/30] Loss: 0.2890  ValAcc: 87.58%
  Epoch [10/30] Loss: 0.2163  ValAcc: 88.90%
  Epoch [15/30] Loss: 0.1454  ValAcc: 89.10%
  Epoch [20/30] Loss: 0.0893  ValAcc: 90.34%
  Epoch [25/30] Loss: 0.0554  ValAcc: 90.34%
  Epoch [30/30] Loss: 0.0451  ValAcc: 90.50%
  Best checkpoint at epoch 29: val_acc=90.50%


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=203, bias=True)
  (2): GELU(approximate='none')
  (3): Dropout(p=0.05, inplace=False)
  (4): Linear(in_features=203, out_features=457, bias=True)
  (5): GELU(approximate='none')
  (6): Dropout(p=0.05, inplace=False)
  (7): Linear(in_features=457, out_features=10, bias=True)
)

In [8]:
# ── Test evaluation ─────────────────────────────────────────────────────────
probas, labels = get_all_probas_and_labels(model, TEST_LOADER, DEVICE, 10)
pr_scores = compute_pr_auc_scores(probas, labels, model_name='optuna_best')
test_acc, cm, per_class = evaluate_detailed(
    model, TEST_LOADER, DEVICE, CLASS_NAMES, model_name='optuna_best')

# Save results
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w') as f:
    json.dump({
        'number': best.number,
        'val_macro_pr_auc': best.value,
        'val_accuracy': best.user_attrs.get('val_acc', None),
        'test_accuracy': test_acc,
        'best_val_epoch': best_epoch,
        'best_val_acc_at_ckpt': best_val_acc,
        'params': best.params,
    }, f, indent=2)

with open(os.path.join(OUT_DIR, 'train_losses_best.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')

torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
print(f'\nAll results saved to {OUT_DIR}/')


  Test Accuracy: 90.00%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8390     0.0177     0.8407
  Trouser             0.9800     0.0004     0.9959
  Pullover            0.8430     0.0207     0.8192
  Dress               0.9080     0.0111     0.9008
  Coat                0.8550     0.0184     0.8374
  Sandal              0.9640     0.0028     0.9747
  Shirt               0.7120     0.0266     0.7487
  Sneaker             0.9620     0.0062     0.9450
  Bag                 0.9760     0.0030     0.9731
  Ankle boot          0.9610     0.0042     0.9620

All results saved to c:\document\Study documents\Deeplearning_Course\outputs\error_analysis\MLP\phase4_optuna/


## Logit bias sweep (same as Phase 1 A3)


In [9]:
SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in TEST_LOADER:
            inputs, lbls = inputs.to(DEVICE), lbls.to(DEVICE)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy()); all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        acc = accuracy_score(all_l, all_p)
        sweep.append({'bias': bias, 'acc': round(acc*100, 2),
                       'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4),
                       'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  '
              f'Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  '
              f'Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n'
            + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} '
                f'{r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')


bias=-1.0  acc=89.93%  Shirt TPR=0.6420  Prec=0.8005
bias=-0.5  acc=90.00%  Shirt TPR=0.6770  Prec=0.7764
bias=+0.0  acc=90.00%  Shirt TPR=0.7120  Prec=0.7487
bias=+0.5  acc=89.87%  Shirt TPR=0.7440  Prec=0.7120
bias=+1.0  acc=89.80%  Shirt TPR=0.7790  Prec=0.6876
bias=+1.5  acc=89.43%  Shirt TPR=0.8120  Prec=0.6501
bias=+2.0  acc=89.05%  Shirt TPR=0.8450  Prec=0.6177


## Comparison vs Phase 1 baseline, Wider, and Deeper


In [10]:
bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
z = [r for r in sweep if r['bias'] == 0.0][0]

print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 56)
print(f'{"Phase 1 baseline":<30} {"90.08":>7} {"0.7070":>9} {"0.7505":>10}')
print(f'{"Exp 2 Wider":<30} {"90.26":>7} {"0.7280":>9} {"0.7599":>10}')
print(f'{"Exp 3 Deeper":<30} {"90.13":>7} {"0.7140":>9} {"0.7645":>10}')
print(f'{"Exp 4 optuna bias=0":<30} {z["acc"]:>7.2f} {z["tpr"]:>9.4f} {z["prec"]:>10.4f}')
print(f'{"Exp 4 bias="+str(bt["bias"])+" (best trade)":<30} '
      f'{bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')


Config                            Acc%  ShirtTPR  ShirtPrec
--------------------------------------------------------
Phase 1 baseline                 90.08    0.7070     0.7505
Exp 2 Wider                      90.26    0.7280     0.7599
Exp 3 Deeper                     90.13    0.7140     0.7645
Exp 4 optuna bias=0              90.00    0.7120     0.7487
Exp 4 bias=2.0 (best trade)      89.05    0.8450     0.6177
